# Document Fraud Detection AI - Google Colab

Runs the **full Document Fraud Detection pipeline** on Colab with free GPU.

**7 Modules:** ELA | CNN (EfficientNet-B3) | ManTraNet | Metadata | OCR | Copy-Move | Blur

---
**Steps:**
1. Runtime > Change runtime type > **T4 GPU**
2. Run cells top to bottom
3. Upload project zip when prompted
4. Test with your documents


## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("No GPU. Go to Runtime > Change runtime type > T4 GPU")


## 2. Upload Project Files

Zip your entire **document_fraud_ai** folder and upload it below.


In [ ]:
import os, zipfile
from google.colab import files

print("Upload your document_fraud_ai.zip ...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Extracting {zip_name}...")

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/")

# Find the project directory
for d in os.listdir("/content/"):
    if os.path.isdir(f"/content/{d}") and os.path.exists(f"/content/{d}/app.py"):
        PROJECT_DIR = f"/content/{d}"
        break
else:
    PROJECT_DIR = "/content/document_fraud_ai"

os.chdir(PROJECT_DIR)
print(f"Project dir: {PROJECT_DIR}")
print(f"Files: {os.listdir(os.curdir)}")


In [ ]:
# === OPTION B: Google Drive (uncomment if needed) ===
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/path/to/document_fraud_ai' /content/
# PROJECT_DIR = '/content/document_fraud_ai'
# os.chdir(PROJECT_DIR)


## 3. Install Dependencies

Colab already has: torch, numpy, scipy, sklearn, PIL, cv2


In [ ]:
!pip install -q \
    fastapi==0.115.6 \
    'uvicorn[standard]==0.34.0' \
    python-multipart==0.0.20 \
    'PyMuPDF>=1.24.0' \
    'loguru>=0.7.0' \
    'pytesseract>=0.3.10' \
    'gdown>=5.1.0' \
    'pyngrok>=7.0.0' \
    'nest_asyncio>=1.6.0'

# Install Tesseract OCR engine (system-level)
!apt-get install -y -q tesseract-ocr > /dev/null 2>&1

print("All dependencies installed!")


## 4. Verify All Modules

In [ ]:
import sys, os
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

checks = []
try:
    import fastapi; checks.append(("FastAPI","OK",fastapi.__version__))
except: checks.append(("FastAPI","FAIL",""))
try:
    import torch; checks.append(("PyTorch","OK",torch.__version__))
except: checks.append(("PyTorch","FAIL",""))
try:
    import torchvision; checks.append(("TorchVision","OK",torchvision.__version__))
except: checks.append(("TorchVision","FAIL",""))
try:
    import fitz; checks.append(("PyMuPDF","OK",fitz.version[0]))
except: checks.append(("PyMuPDF","FAIL",""))
try:
    import cv2; checks.append(("OpenCV","OK",cv2.__version__))
except: checks.append(("OpenCV","FAIL",""))
try:
    from sklearn.cluster import DBSCAN; checks.append(("scikit-learn","OK",""))
except: checks.append(("scikit-learn","FAIL",""))
try:
    import loguru; checks.append(("Loguru","OK",""))
except: checks.append(("Loguru","FAIL",""))
try:
    import pytesseract; pytesseract.get_tesseract_version(); checks.append(("Tesseract","OK",""))
except: checks.append(("Tesseract","FAIL",""))

for name,status,ver in checks:
    icon = "OK" if status=="OK" else "XX"
    print(f"[{icon}] {name:<18} {status:<8} {ver}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")


## 5. Initialize Fraud Detection Pipeline

If you have a trained model (.pth), set MODEL_PATH below.


In [ ]:
import sys, os, torch
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from fraud_model.pipeline import FraudDetectionPipeline

MODEL_PATH = None  # e.g. "/content/best_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

pipeline = FraudDetectionPipeline(model_path=MODEL_PATH, device=device)

print(f"Pipeline ready on: {device}")
print(f"CNN model loaded: {pipeline.model_loaded}")
print(f"ManTraNet loaded: {pipeline.mantranet_model is not None}")


## 6. Start FastAPI Server + Public URL

Runs the API in background and creates a public URL via ngrok.

Optional: get a free ngrok token at https://ngrok.com


In [ ]:
import nest_asyncio, threading, time, uvicorn, sys, os, json
nest_asyncio.apply()

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from app import app as fastapi_app
import app as app_module
app_module.pipeline = pipeline

def run_server():
    uvicorn.run(fastapi_app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("FastAPI server running on port 8000")

try:
    from pyngrok import ngrok
    # ngrok.set_auth_token("YOUR_TOKEN")  # optional for longer sessions
    public_url = ngrok.connect(8000)
    print(f"\nPublic API URL: {public_url}")
    print(f"Swagger docs:  {public_url}/docs")
except Exception as e:
    print(f"ngrok failed: {e}")
    public_url = "http://localhost:8000"

# Quick health check
import urllib.request
try:
    resp = urllib.request.urlopen("http://localhost:8000/health")
    print(f"\nHealth: {json.loads(resp.read())}")
except Exception as e:
    print(f"Health check failed: {e}")


## 7. Analyze a Document (Upload & Test)

Upload a PDF, JPG, or PNG to analyze for fraud.


In [ ]:
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
import json, time, os

print("Upload a document to analyze...")
uploaded = files.upload()

for filename, content in uploaded.items():
    os.makedirs("uploads", exist_ok=True)
    filepath = os.path.join("uploads", filename)
    with open(filepath, "wb") as f:
        f.write(content)

    print(f"\nAnalyzing: {filename} ({len(content)/1024:.1f} KB)")

    ext = os.path.splitext(filename)[1].lower()
    if ext in [".jpg",".jpeg",".png",".bmp",".webp"]:
        display(IPImage(filename=filepath, width=400))

    start = time.time()
    result = pipeline.analyze(filepath)
    elapsed = time.time() - start

    fp = result["fraud_probability"]
    fraud = result["is_fraud"]
    conf = result["confidence"]
    color = "red" if fraud else "green"
    verdict = "FRAUD DETECTED" if fraud else "GENUINE"

    display(HTML(
        f'<div style="padding:15px;border:3px solid {color};border-radius:10px;margin:10px 0">'
        f'<h2 style="color:{color};margin:0">{verdict}</h2>'
        f'<p>Fraud Probability: <b>{fp:.1%}</b></p>'
        f'<p>Confidence: <b>{conf:.1f}%</b> | Time: {elapsed:.2f}s</p>'
        f'</div>'
    ))

    print("\nModule Scores:")
    scores = result.get("details",{}).get("module_scores",{})
    for mod, sc in scores.items():
        if sc is not None:
            bar = "#" * int(sc * 30)
            print(f"  {mod:<20} {sc:.4f} |{bar}")
        else:
            print(f"  {mod:<20} N/A")

    print("\nAnomalies:")
    for r in result.get("reasons",[]):
        print(f"  - {r}")

    os.remove(filepath)
    print("\n" + "=" * 60)


## 8. Test via REST API

In [ ]:
# Check model info
!curl -s http://localhost:8000/model/info | python -m json.tool


## 9. Batch Analysis (Multiple Documents)

In [ ]:
from google.colab import files
import time, os

print("Upload multiple documents...")
uploaded = files.upload()
os.makedirs("uploads", exist_ok=True)

for filename, content in uploaded.items():
    filepath = os.path.join("uploads", filename)
    with open(filepath, "wb") as f:
        f.write(content)

    start = time.time()
    result = pipeline.analyze(filepath)
    elapsed = time.time() - start

    status = "FRAUD" if result["is_fraud"] else "OK"
    prob = result['fraud_probability']
    conf = result['confidence']
    print(f"  [{status:>5}] {filename:<40} prob={prob:.4f}  conf={conf:.1f}%  ({elapsed:.2f}s)")
    os.remove(filepath)

print("\nDone!")


## 10. (Optional) Setup ManTraNet

Adds pixel-level forgery detection (25% weight, 385 manipulation types).

Get weights from: https://github.com/RonyAbecidan/ManTraNet-pytorch


In [ ]:
# Uncomment to enable ManTraNet:

# from utils.mantranet import setup_mantranet

# Option 1: Weights on Google Drive
# setup_mantranet(weights_gdrive_id="YOUR_GDRIVE_FILE_ID")

# Option 2: Weights uploaded to Colab
# setup_mantranet(weights_path="/content/ManTraNet.pt")

# Then reinitialize pipeline:
# pipeline = FraudDetectionPipeline(model_path=MODEL_PATH, device=device)
# print(f"ManTraNet loaded: {pipeline.mantranet_model is not None}")

print("ManTraNet setup cell - uncomment to enable")


## 11. (Optional) Train Custom Model

Dataset structure:
```
dataset/
  genuine/    <- real documents
  tampered/   <- forged documents
```


In [ ]:
# Upload dataset to /content/dataset/ first, then uncomment:

# !python train_model.py \
#     --data_dir /content/dataset \
#     --batch_size 4 \
#     --accum_steps 8 \
#     --epochs 50 \
#     --freeze_epochs 5

# After training, reload pipeline:
# pipeline = FraudDetectionPipeline(model_path="best_model.pth", device=device)

print("Training cell - uncomment to train")


## 12. (Optional) Launch Streamlit Web UI

Starts the full drag-and-drop web frontend.


In [ ]:
!pip install -q streamlit
!npm install -g localtunnel > /dev/null 2>&1

# Write a helper script to launch streamlit
with open('launch_streamlit.sh', 'w') as f:
    f.write('streamlit run streamlit_app.py --server.port 8501 --server.headless true --server.address 0.0.0.0 &\n')
    f.write('sleep 5\n')
    f.write('npx localtunnel --port 8501\n')

!bash launch_streamlit.sh
